# Khmer OCR three architectures
CPU demo with real predictions. These examples were selected for Transformer success, not for a fair accuracy comparison. Set TRY_YOUR_IMAGE=True in the final cell to try your own cropped line.

In [ ]:
%pip -q install torch pillow numpy huggingface_hub
from huggingface_hub import hf_hub_download
from pathlib import Path
import sys, zipfile, json
repo='Seoul12/khmer-ocr-three-architectures'
module=hf_hub_download(repo,'inference_three.py')
sys.path.insert(0,str(Path(module).parent))
from inference_three import MultiRecognizer
archive=hf_hub_download(repo,'demo_dataset.zip')
with zipfile.ZipFile(archive) as z: z.extractall('selected_demo')
manifest=next(Path('selected_demo').rglob('manifest.json'))
rows=json.loads(manifest.read_text())
models={k:MultiRecognizer(hf_hub_download(repo,k+'.pt')) for k in ['bilstm','bigru','transformer']}


In [ ]:
from PIL import Image
from IPython.display import display
scores={k:0 for k in models}
for row in rows:
    image=manifest.parent/row['image']
    display(Image.open(image))
    print('ANSWER:',row['answer'])
    for name,recognizer in models.items():
        prediction=recognizer.predict(image)
        correct=prediction==row['answer']; scores[name]+=correct
        print(name,'RIGHT' if correct else 'WRONG','PREDICTION:',prediction)
print('Selected demonstration results:',scores,'out of',len(rows))
print('Selected for Transformer success. This is not a fair benchmark.')


In [ ]:
TRY_YOUR_IMAGE=False
if TRY_YOUR_IMAGE:
    from google.colab import files
    for filename in files.upload():
        display(Image.open(filename))
        for name,recognizer in models.items(): print(name,recognizer.predict(filename))
